# FIFA World Cup 2026 - Fantasy Football Data

Data sourced directly from the official FIFA Fantasy API:
- `https://play.fifa.com/json/fantasy/players.json` - All 1248 active players with positions & prices
- `https://play.fifa.com/json/fantasy/squads.json` - All 48 teams with groups
- `https://play.fifa.com/json/fantasy/rounds.json` - Tournament rounds/schedule

**TODO**: Player stats (goals, assists, games) for club + national team over last 4 seasons.

In [1]:
import requests
import json
import polars as pl
import pandas as pd
from pathlib import Path
import analytics as an  # shared analytics (also used by app.py)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36'
}
BASE_URL = 'https://play.fifa.com/json/fantasy'

print('Ready')

Ready


## 1. Fetch data from FIFA Fantasy API

In [2]:
def fetch_and_cache(endpoint, filename):
    """Fetch from API if not cached locally."""
    path = Path(filename)
    if path.exists():
        print(f'Loading cached {filename}')
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    
    print(f'Fetching {endpoint}...')
    resp = requests.get(f'{BASE_URL}/{endpoint}', headers=HEADERS, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f'Saved to {filename}')
    return data

raw_players = fetch_and_cache('players.json', 'fifa_fantasy_players.json')
raw_squads = fetch_and_cache('squads.json', 'fifa_fantasy_squads.json')
raw_rounds = fetch_and_cache('rounds.json', 'fifa_fantasy_rounds.json')

print(f'\nPlayers: {len(raw_players)}  |  Squads: {len(raw_squads)}  |  Rounds: {len(raw_rounds)}')

Loading cached fifa_fantasy_players.json
Loading cached fifa_fantasy_squads.json
Loading cached fifa_fantasy_rounds.json

Players: 1481  |  Squads: 48  |  Rounds: 8


## 2. Build squad lookup & player DataFrame

In [3]:
# Squad ID -> country name + group
squad_map = {s['id']: s['name'] for s in raw_squads}
squad_group = {s['id']: s['group'].upper() for s in raw_squads}
squad_abbr = {s['id']: s['abbr'] for s in raw_squads}

# Build flat player records
rows = []
for p in raw_players:
    name = p['knownName'] or f"{p['firstName']} {p['lastName']}"
    rows.append({
        'player_id': p['id'],
        'fifa_id': p.get('fifaId'),
        'first_name': p['firstName'],
        'last_name': p['lastName'],
        'known_name': p['knownName'],
        'display_name': name.strip(),
        'country': squad_map.get(p['squadId'], 'Unknown'),
        'country_abbr': squad_abbr.get(p['squadId'], ''),
        'group': squad_group.get(p['squadId'], ''),
        'squad_id': p['squadId'],
        'fantasy_position': p['position'],       # GK, DEF, MID, FWD
        'fantasy_price': p['price'],              # in $M
        'status': p['status'],                    # playing / transferred
        'percent_selected': p['percentSelected'],
        'total_points': p['stats']['totalPoints'],
        'avg_points': p['stats']['avgPoints'],
        'form': p['stats']['form'],
        'one_to_watch': p['oneToWatch'],
    })

df_all = pl.DataFrame(rows)
df_players = df_all.filter(pl.col('status') == 'playing')

print(f'Total in API: {len(df_all)}  |  Active (playing): {len(df_players)}  |  Transferred: {len(df_all) - len(df_players)}')
print(f'Countries: {df_players["country"].n_unique()}')
print(f'\nPosition breakdown:')
df_players.group_by('fantasy_position').len().sort('fantasy_position')

Total in API: 1481  |  Active (playing): 1248  |  Transferred: 233
Countries: 48

Position breakdown:


fantasy_position,len
str,u32
"""DEF""",412
"""FWD""",269
"""GK""",145
"""MID""",422


## 3. Explore the data

In [4]:
# Price distribution
print('=== PRICE STATS ===')
print(df_players.select('fantasy_price').describe())
print(f'\nPrice range: ${df_players["fantasy_price"].min()}m - ${df_players["fantasy_price"].max()}m')

print('\n=== MOST EXPENSIVE PLAYERS ===')
top_price = df_players.sort('fantasy_price', descending=True).head(20)
top_price.select(['display_name', 'country', 'fantasy_position', 'fantasy_price', 'percent_selected'])

=== PRICE STATS ===
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ fantasy_price │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 1248.0        │
│ null_count ┆ 0.0           │
│ mean       ┆ 4.894631      │
│ std        ┆ 1.164248      │
│ min        ┆ 3.5           │
│ 25%        ┆ 4.0           │
│ 50%        ┆ 4.7           │
│ 75%        ┆ 5.6           │
│ max        ┆ 10.5          │
└────────────┴───────────────┘

Price range: $3.5m - $10.5m

=== MOST EXPENSIVE PLAYERS ===


display_name,country,fantasy_position,fantasy_price,percent_selected
str,str,str,f64,f64
"""Harry Kane""","""England""","""FWD""",10.5,37.9
"""Kylian Mbappé""","""France""","""FWD""",10.5,51.0
"""Erling Haaland""","""Norway""","""FWD""",10.5,33.2
"""Lionel Messi""","""Argentina""","""FWD""",10.0,18.2
"""Vinícius Júnior""","""Brazil""","""MID""",10.0,14.2
…,…,…,…,…
"""Raphinha""","""Brazil""","""MID""",8.2,21.4
"""Luis Díaz""","""Colombia""","""MID""",8.1,18.7
"""Mikel Oyarzabal""","""Spain""","""FWD""",8.1,16.5


In [5]:
# Most selected players (proxy for starting XI likelihood)
print('=== MOST SELECTED (likely starters) ===')
top_selected = df_players.sort('percent_selected', descending=True).head(30)
top_selected.select(['display_name', 'country', 'fantasy_position', 'fantasy_price', 'percent_selected'])

=== MOST SELECTED (likely starters) ===


display_name,country,fantasy_position,fantasy_price,percent_selected
str,str,str,f64,f64
"""Kylian Mbappé""","""France""","""FWD""",10.5,51.0
"""Bruno Fernandes""","""Portugal""","""MID""",8.5,48.8
"""Lamine Yamal""","""Spain""","""MID""",10.0,44.6
"""Nuno Mendes""","""Portugal""","""DEF""",5.8,43.5
"""Harry Kane""","""England""","""FWD""",10.5,37.9
…,…,…,…,…
"""Nico O'Reilly""","""England""","""DEF""",4.7,13.2
"""Manuel Neuer""","""Germany""","""GK""",5.0,13.2
"""Cristiano Ronaldo""","""Portugal""","""FWD""",10.0,13.0


In [6]:
# Players per country
print('=== SQUAD SIZES ===')
print(df_players.group_by('country').len().sort('country'))

=== SQUAD SIZES ===
shape: (48, 2)
┌────────────┬─────┐
│ country    ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ Algeria    ┆ 26  │
│ Argentina  ┆ 26  │
│ Australia  ┆ 26  │
│ Austria    ┆ 25  │
│ Belgium    ┆ 26  │
│ …          ┆ …   │
│ Tunisia    ┆ 26  │
│ Türkiye    ┆ 26  │
│ USA        ┆ 26  │
│ Uruguay    ┆ 26  │
│ Uzbekistan ┆ 26  │
└────────────┴─────┘


In [7]:
# Full squad for a sample country
sample_country = 'France'
print(f'=== {sample_country.upper()} SQUAD ===')
squad = df_players.filter(pl.col('country') == sample_country).sort(['fantasy_position', 'fantasy_price'], descending=[False, True])
squad.select(['display_name', 'fantasy_position', 'fantasy_price', 'percent_selected'])

=== FRANCE SQUAD ===


display_name,fantasy_position,fantasy_price,percent_selected
str,str,f64,f64
"""Jules Koundé""","""DEF""",5.4,7.4
"""William Saliba""","""DEF""",5.3,15.8
"""Dayot Upamecano""","""DEF""",5.3,4.9
"""Lucas Hernández""","""DEF""",5.2,2.5
"""Malo Gusto""","""DEF""",5.1,0.5
…,…,…,…
"""Adrien Rabiot""","""MID""",6.4,0.2
"""Maghnes Akliouche""","""MID""",6.4,0.0
"""Warren Zaïre-Emery""","""MID""",6.1,0.3


## 4. Starting XI likelihood

We use `percent_selected` as a proxy: within each country, players with higher selection % are more likely starters. We rank within each squad by position.

In [8]:
# Typical formation: 1 GK, 4 DEF, 4 MID, 2 FWD = 11 starters
STARTING_SLOTS = {'GK': 1, 'DEF': 4, 'MID': 4, 'FWD': 2}

df_ranked = df_players.with_columns(
    pl.col('percent_selected')
      .rank(method='ordinal', descending=True)
      .over(['country', 'fantasy_position'])
      .alias('pos_rank')
)

def classify_starter(pos, rank):
    slots = STARTING_SLOTS.get(pos, 2)
    if rank <= slots:
        return 'Likely Starter'
    elif rank <= slots + 2:
        return 'Maybe Starter'
    else:
        return 'Unlikely Starter'

df_ranked = df_ranked.with_columns(
    pl.struct(['fantasy_position', 'pos_rank'])
      .map_elements(lambda x: classify_starter(x['fantasy_position'], x['pos_rank']), return_dtype=pl.Utf8)
      .alias('starting_likelihood')
)

print('Starting likelihood distribution:')
print(df_ranked.group_by('starting_likelihood').len().sort('starting_likelihood'))
print(f'\n=== FRANCE STARTING XI ===')
france_xi = df_ranked.filter((pl.col('country') == 'France') & (pl.col('starting_likelihood') == 'Likely Starter'))
france_xi.sort('fantasy_position').select(['display_name', 'fantasy_position', 'fantasy_price', 'percent_selected'])

Starting likelihood distribution:
shape: (3, 2)
┌─────────────────────┬─────┐
│ starting_likelihood ┆ len │
│ ---                 ┆ --- │
│ str                 ┆ u32 │
╞═════════════════════╪═════╡
│ Likely Starter      ┆ 528 │
│ Maybe Starter       ┆ 377 │
│ Unlikely Starter    ┆ 343 │
└─────────────────────┴─────┘

=== FRANCE STARTING XI ===


display_name,fantasy_position,fantasy_price,percent_selected
str,str,f64,f64
"""Theo Hernández""","""DEF""",5.0,3.3
"""Dayot Upamecano""","""DEF""",5.3,4.9
"""William Saliba""","""DEF""",5.3,15.8
"""Jules Koundé""","""DEF""",5.4,7.4
"""Kylian Mbappé""","""FWD""",10.5,51.0
…,…,…,…
"""Mike Maignan""","""GK""",5.0,10.3
"""Ousmane Dembélé""","""MID""",10.0,20.2
"""Désiré Doué""","""MID""",7.5,3.3


## 5. Export complete dataset

In [9]:
# Final DataFrame
df_final = df_ranked.select([
    'player_id', 'fifa_id', 'display_name', 'first_name', 'last_name',
    'country', 'country_abbr', 'group', 'squad_id',
    'fantasy_position', 'fantasy_price', 'percent_selected',
    'starting_likelihood', 'pos_rank',
    'total_points', 'avg_points', 'form', 'one_to_watch',
]).sort(['group', 'country', 'fantasy_position', 'fantasy_price'], descending=[False, False, False, True])

# Add placeholder columns for stats (to be filled later)
df_final = df_final.with_columns([
    pl.lit(None).cast(pl.Int64).alias('club_goals_4y'),
    pl.lit(None).cast(pl.Int64).alias('club_assists_4y'),
    pl.lit(None).cast(pl.Int64).alias('club_games_4y'),
    pl.lit(None).cast(pl.Int64).alias('nt_goals_4y'),
    pl.lit(None).cast(pl.Int64).alias('nt_assists_4y'),
    pl.lit(None).cast(pl.Int64).alias('nt_games_4y'),
])

# Save
df_final.write_csv('wc2026_players.csv')
df_final.write_parquet('wc2026_players.parquet')

print(f'Saved {len(df_final)} players to wc2026_players.csv / .parquet')
print(f'\n=== DATASET SUMMARY ===')
print(f'Players:    {len(df_final)}')
print(f'Countries:  {df_final["country"].n_unique()}')
print(f'Groups:     {df_final["group"].n_unique()}')
print(f'Positions:  {df_final["fantasy_position"].value_counts().sort("fantasy_position")}')
print(f'Price range: ${df_final["fantasy_price"].min()}m - ${df_final["fantasy_price"].max()}m')
print(f'\nColumns: {df_final.columns}')

Saved 1248 players to wc2026_players.csv / .parquet

=== DATASET SUMMARY ===
Players:    1248
Countries:  48
Groups:     12
Positions:  shape: (4, 2)
┌──────────────────┬───────┐
│ fantasy_position ┆ count │
│ ---              ┆ ---   │
│ str              ┆ u32   │
╞══════════════════╪═══════╡
│ DEF              ┆ 412   │
│ FWD              ┆ 269   │
│ GK               ┆ 145   │
│ MID              ┆ 422   │
└──────────────────┴───────┘
Price range: $3.5m - $10.5m

Columns: ['player_id', 'fifa_id', 'display_name', 'first_name', 'last_name', 'country', 'country_abbr', 'group', 'squad_id', 'fantasy_position', 'fantasy_price', 'percent_selected', 'starting_likelihood', 'pos_rank', 'total_points', 'avg_points', 'form', 'one_to_watch', 'club_goals_4y', 'club_assists_4y', 'club_games_4y', 'nt_goals_4y', 'nt_assists_4y', 'nt_games_4y']


In [10]:
 # Quick view
df_final.head(20)

player_id,fifa_id,display_name,first_name,last_name,country,country_abbr,group,squad_id,fantasy_position,fantasy_price,percent_selected,starting_likelihood,pos_rank,total_points,avg_points,form,one_to_watch,club_goals_4y,club_assists_4y,club_games_4y,nt_goals_4y,nt_assists_4y,nt_games_4y
i64,null,str,str,str,str,str,str,i64,str,f64,f64,str,u32,i64,i64,i64,bool,i64,i64,i64,i64,i64,i64
374,null,"""Ladislav Krejcí""","""Ladislav""","""Krejcí""","""Czechia""","""CZE""","""A""",15,"""DEF""",4.4,0.6,"""Likely Starter""",2,0,0,0,false,null,null,null,null,null,null
378,null,"""Robin Hranác""","""Robin""","""Hranác""","""Czechia""","""CZE""","""A""",15,"""DEF""",3.9,0.0,"""Maybe Starter""",6,0,0,0,false,null,null,null,null,null,null
1405,null,"""David Zima""","""David""","""Zima""","""Czechia""","""CZE""","""A""",15,"""DEF""",3.9,0.0,"""Unlikely Starter""",8,0,0,0,false,null,null,null,null,null,null
379,null,"""Jaroslav Zeleny""","""Jaroslav""","""Zeleny""","""Czechia""","""CZE""","""A""",15,"""DEF""",3.9,0.0,"""Unlikely Starter""",7,0,0,0,false,null,null,null,null,null,null
375,null,"""Tomás Holes""","""Tomás""","""Holes""","""Czechia""","""CZE""","""A""",15,"""DEF""",3.9,0.0,"""Likely Starter""",4,0,0,0,false,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
388,null,"""Lukás Hornícek""","""Lukás""","""Hornícek""","""Czechia""","""CZE""","""A""",15,"""GK""",3.8,0.1,"""Maybe Starter""",3,0,0,0,false,null,null,null,null,null,null
397,null,"""Pavel Sulc""","""Pavel""","""Sulc""","""Czechia""","""CZE""","""A""",15,"""MID""",5.9,0.2,"""Likely Starter""",2,0,0,0,false,null,null,null,null,null,null
396,null,"""Lukás Provod""","""Lukás""","""Provod""","""Czechia""","""CZE""","""A""",15,"""MID""",5.9,0.0,"""Unlikely Starter""",8,0,0,0,false,null,null,null,null,null,null


## 6. Player stats (API-Football) joined with fantasy data

Loads the joined stats via the shared **`analytics.load_data()`** (the same function `app.py`
uses), so the notebook and app stay in sync. Returns `stats` (one row per player / competition /
season, with fantasy attributes) plus `team_stats`, `team_season`, `players`.

In [ ]:
# Load everything via the shared module (analytics.py) - also used by app.py.
stats, team_stats, team_season, players = an.load_data()

print(f'{len(stats):,} rows x {stats.width} cols - {stats["player_id"].n_unique()} players')
stats

In [17]:
#unique fantasy positions:
stats['fantasy_position'].value_counts()

fantasy_position,count
str,u32
"""MID""",7762
"""FWD""",4715
"""DEF""",7047
"""GK""",1903


In [15]:
stats.filter(pl.col("display_name") == 'Rayan Aït-Nouri')

player_id,apifootball_id,display_name,country,competition,comp_type,team_name,season,season_start_year,games,started,minutes,goals,assists,tackles,interceptions,duels,duels_won,rating,fantasy_position,fantasy_price,group,starting_likelihood
i64,i64,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,str,f64,str,str
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""League Cup""","""club""","""Wolves""","""2022/23""",2022,4,3,248,1,1,12,5,48,24,7.45,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""Premier League""","""club""","""Wolves""","""2022/23""",2022,21,9,1068,1,null,33,9,152,86,6.628571,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""League Cup""","""club""","""Manchester City""","""2022/23""",2022,4,3,248,1,1,12,5,48,24,7.45,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""FA Cup""","""club""","""Wolves""","""2022/23""",2022,2,2,163,0,null,2,2,21,9,6.25,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""Friendlies Clubs""","""club""","""Wolves""","""2022/23""",2022,1,1,86,0,null,null,null,null,null,null,"""DEF""",4.9,"""J""","""Likely Starter"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""Africa Cup of Nations""","""national""","""Algeria""","""2025/26""",2025,5,5,356,0,0,10,4,53,25,6.84,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""League Cup""","""club""","""Manchester City""","""2025/26""",2025,2,1,165,0,2,5,5,19,12,7.03,"""DEF""",4.9,"""J""","""Likely Starter"""
1,21138,"""Rayan Aït-Nouri""","""Algeria""","""FA Cup""","""club""","""Manchester City""","""2025/26""",2025,2,2,135,0,0,null,2,15,7,7.2,"""DEF""",4.9,"""J""","""Likely Starter"""


## 7. National team records

`team_stats` (per team / season / competition) and `team_season` (per team / season) were loaded
in Section 6. `team_totals` below uses the shared **`analytics.per_country_stats`** - 4-season
totals with per-game rates.

**Note:** raw totals favour teams that play more matches (African sides with AFCON + qualifiers
+ friendlies). Compare with `win_rate` / per-game rates, and filter `competition` to drop
`Friendlies` for competitive-only views.

In [ ]:
# 4-season national-team totals with per-game rates (shared analytics.per_country_stats)
team_totals = an.per_country_stats(team_season)

print(f'{team_stats["country"].n_unique()} teams - {len(team_stats)} detail rows - {len(team_season)} season rows')
team_totals

## 8. Player ranking (shared `analytics.rank_players`)

`an.rank_players(stats, position, metric=..., years=..., comp_type=..., country=...,
starting_likelihood=..., min_price=..., max_price=..., score_mode='per_game', min_games=0, top=20)`
returns a sorted table, highest score on top. Same function powers `app.py`.

- **position**: one (`'FWD'`) or several (`['FWD','MID']`). Position-dependent metrics score each player by their own position.
- **metric** (default `'real multiplication of goals and assists'`): `'sum goals and assists'`,
  `'goals only'`, `'assists only'`, `'interceptions'`,
  `'weighted average interceptions, assists, goals'` (0.05/0.25/0.65), `'saves'`,
  `'real multiplication of goals and assists'` (assists x3 all; goals x4 FWD / x5 MID / x6 DEF / x7 GK).
  Plain-language formulas are in `an.METRIC_FORMULAS` (also shown in the app expander).
- **years**: `[start, end]` inclusive (default all). **comp_type**: `None`/`'club'`/`'national'` (affects score only).
- **country**, **starting_likelihood**: name or list (optional). **min_price/max_price**: market-value ($M) range.
- **score_mode**: `'per_game'` (default) or `'total'`. **min_games**: drop small samples. **top**: rows (`None`=all).

Columns: player, nation, position, club, market_value, starting_likelihood, club_/nt_ games/goals/assists,
interceptions, saves (if GK), score.

In [ ]:
# rank_players lives in analytics.py (shared with app.py). Examples:
#   an.rank_players(stats, 'MID', 'assists only', comp_type='national')
#   an.rank_players(stats, 'GK', 'saves', years=[2025, 2025])
#   an.rank_players(stats, 'DEF', country=['Spain', 'Morocco'], top=15)
an.rank_players(stats, 'FWD')